**### ** Bronzework Incremental****

## **Step1 - Import and Setup**

In [0]:
from pyspark.sql import functions as F   # PySpark built-in SQL functions (col, lit, max, current_timestamp, etc.)
from delta.tables import DeltaTable      # Delta Lake API — used for MERGE / upsert operations on Delta tables
from datetime import datetime            # Standard library — generates UTC timestamps written to audit columns
import uuid                              # Standard library — generates unique UUID v4 run identifiers

In [0]:
# Set the active Unity Catalog to 'databricks_dev'.
# All subsequent unqualified schema/table references (e.g. 00_bronze.*) resolve within this catalog.
spark.sql("use catalog databricks_dev ")

In [0]:
# Create the '00_bronze' schema inside the active catalog if it does not already exist.
# This schema is the landing zone for all raw Bronze Delta tables written by this pipeline.
spark.sql("create schema if not exists 00_bronze")


### **Step 2 - Bronze control table**
 This table stores the **watermark** for each source table.
-  the latest timestamp already processed
- the latest primary key processed at that timestamp
- how many rows were written in the latest run

This is what makes the Bronze load incremental and rerun safe.

In [0]:
# Create the Bronze ingestion control table if it does not already exist.
# This table stores one watermark record per source table and is the foundation for:
#   1. Incremental loading  — only rows newer than the last watermark are processed each run
#   2. Idempotent reruns    — re-running from the stored watermark produces the same result
#   3. Audit / observability — last run ID, row count, and status are persisted after each run
spark.sql("""
CREATE TABLE IF NOT EXISTS 00_bronze.ingestion_control (
    layer               STRING,    -- Pipeline layer this record belongs to (always 'bronze')
    table_name          STRING,    -- Name of the source table being tracked
    ts_col              STRING,    -- Watermark / timestamp column name in the source table
    pk_col              STRING,    -- Primary key column name in the source table
    last_successful     TIMESTAMP, -- Max timestamp of a row loaded in the last successful run
    last_successful_pk  BIGINT,    -- PK of the last row loaded at last_successful (tiebreaker)
    last_run_id         STRING,    -- UUID identifying the pipeline run that wrote this record
    rows_written        LONG,      -- Number of rows appended during the last successful run
    run_status          STRING,    -- Outcome of the last run: 'success' or 'failed'
    updated_ts          TIMESTAMP  -- UTC time this control record was last updated
) USING DELTA
""")

**Step 3 - Source table configuration**
This cell defines which source tables will be loaded into bronze and which columns should be used as:
- primary key
- timestamp/watermark column

> It will also creates a unique **bronze_run_id** for the current pipeline run

In [0]:
# Configuration registry for every source table to be ingested into the Bronze layer.
# Each key is the source table name; each value specifies:
#   ts_col : watermark / timestamp column used for incremental row filtering
#   pk_col : primary key column used as a tiebreaker when two rows share the same timestamp
tables_config = {
    "orders":   {"ts_col": "updated_at",   "pk_col": "order_id"},    # Orders transactional table
    "products": {"ts_col": "updated_at",   "pk_col": "product_id"},  # Products dimension table
    "payments": {"ts_col": "processed_at", "pk_col": "payment_id"},  # Payments transactional table
}

# Generate a unique UUID v4 as the run identifier for this Bronze pipeline execution.
# Every row written during this run is tagged with this ID, enabling end-to-end lineage
# and making it straightforward to audit or correlate rows from a specific run.
bronze_run_id = str(uuid.uuid4())
print(f"bronze run id: {bronze_run_id}")


    

**### Step 4 - Helper functions**                                                  
This cell contains reusable functions:
- **get_last_successful_watermark**() reads the last processed timestamp/watermark from the control 
- **upsert_bronze_control()** updates the control table after a successful Bronze 
load

These functions keep the main logic cleaner and easier to underatnd.

In [0]:
def get_last_successful_watermark(table_name: str):
    """
    Retrieve the most recent successful watermark for a given source table.

    Queries the Bronze ingestion control table and returns the latest
    (timestamp, primary_key) pair from a successful run. These values drive
    the incremental filter in the main load loop so that only new or changed
    rows are read from the source.

    Parameters
    ----------
    table_name : str
        Name of the source table whose watermark is being retrieved.

    Returns
    -------
    tuple : (last_successful_watermark, last_successful_pk)
        last_successful_watermark : datetime | None
            Timestamp of the last row loaded in a successful run.
            Returns None if no prior successful run exists — triggers a full initial load.
        last_successful_pk : int | None
            Primary key of the last row loaded at last_successful_watermark.
            Used as a tiebreaker to avoid duplicating rows that share the boundary timestamp.
            Returns None if no prior successful run exists.
    """
    ctrl = (
        spark.table("00_bronze.ingestion_control")
        .filter(F.col("layer") == "bronze")           # Scope to the Bronze layer only
        .filter(F.col("table_name") == table_name)    # Scope to the requested source table
        .filter(F.col("run_status") == "success")     # Ignore failed or in-progress run records
        .orderBy(F.col("updated_ts").desc())           # Latest successful run first
        .limit(1)                                      # Keep only the single most recent record
        .select(
            F.col("last_successful").alias("last_successful_watermark"),
            F.col("last_successful_pk")
        )
    )
    rows = ctrl.collect()                              # Materialise — at most 1 row returned
    if len(rows) == 0:
        # No successful run found; signal a full initial load by returning None for both values
        return None, None
    else:
        # Return watermark timestamp and PK from the most recent successful run
        return rows[0]["last_successful_watermark"], rows[0]["last_successful_pk"]

In [0]:
def upsert_bronze_control(table_name, ts_col, pk_col, last_ts, last_pk, rows_written, run_id):
    """
    Persist watermark metadata to the ingestion control table after a successful Bronze load.

    Constructs a single-row DataFrame reflecting the new watermark state and merges it into
    the control table on the composite key (layer, table_name).
      - First run for a table  → inserts a new control row.
      - Subsequent runs        → updates the existing row in place.
    This maintains exactly one control record per source table at all times.

    Parameters
    ----------
    table_name   : str      Name of the source table that was loaded.
    ts_col       : str      Watermark / timestamp column name used for this table.
    pk_col       : str      Primary key column name used for this table.
    last_ts      : datetime Maximum timestamp observed in the rows written this run.
    last_pk      : int      Maximum primary key at last_ts — used as a rerun tiebreaker.
    rows_written : int      Number of rows appended to the Bronze Delta table this run.
    run_id       : str      UUID of the current pipeline run for lineage tracking.
    """
    # Build a single-row DataFrame capturing the updated watermark state for this table
    control_df = (
        spark.createDataFrame(
            [(
                "bronze",                                          # layer
                table_name,                                        # table_name
                ts_col,                                            # ts_col
                pk_col,                                            # pk_col
                last_ts,                                           # last_successful
                int(last_pk) if last_pk is not None else None,     # last_successful_pk (None-safe cast)
                run_id,                                            # last_run_id
                int(rows_written),                                 # rows_written (explicit int cast)
                "success",                                         # run_status
                datetime.utcnow(),                                 # updated_ts (current UTC timestamp)
            )],
            schema=(
                "layer STRING, table_name STRING, ts_col STRING, pk_col STRING, "
                "last_successful TIMESTAMP, last_successful_pk BIGINT, last_run_id STRING, "
                "rows_written LONG, run_status STRING, updated_ts TIMESTAMP"
            ),
        )
    )

    # MERGE into the control table matching on (layer, table_name):
    #   WHEN MATCHED     → overwrite all columns with the latest run's values
    #   WHEN NOT MATCHED → insert a new row for tables appearing for the first time
    (
        DeltaTable.forName(spark, "00_bronze.ingestion_control")
        .alias("t")
        .merge(control_df.alias("s"), "t.table_name = s.table_name AND t.layer = s.layer")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )




**### Step 5 - Bronze incremental load loop**
This is the main Bronze logic.

For each table, the notebook:

- Reads the latest timestamp/watermark
- reads the source SQl table
- filters only **new/changed rows**
- adds Bronze audit columns
- appends the rows into the Bronze Delta table
- updates the control table

This is the core incremental loading logic

In [0]:
import pyspark.sql.functions as F  # Re-import ensures F is available even when this cell runs standalone
from datetime import datetime      # Re-import for UTC timestamp generation within audit columns

# ── Main Bronze incremental load loop ───────────────────────────────────────
# Iterates over every table defined in tables_config.
# For each table the loop:
#   1. Reads the last successful watermark (timestamp + PK) from the ingestion control table
#   2. Reads the source SQL Server table via the Lakehouse Federation catalog
#   3. Filters only rows newer than the watermark (or all rows on the very first run)
#   4. Appends qualifying rows to the Bronze Delta table with three audit columns
#   5. Updates the ingestion control table with the new watermark and run metadata
for table_name, table_config in tables_config.items():
    ts_col = table_config["ts_col"]  # Watermark column name for this table
    pk_col = table_config["pk_col"]  # Primary key column name for this table
    source_table = f"`azuresqlserver-connection_catalog`.dbo.{table_name}"  # Fully qualified Lakehouse Federation source
    target_table = f"00_bronze.{table_name}_raw"                            # Target Bronze Delta table

    # Step 1 — Fetch the latest watermark from the ingestion control table
    last_successful, last_successful_pk = get_last_successful_watermark(table_name)

    # Normalise the watermark timestamp to millisecond precision to match SQL Server's datetime2 accuracy
    if last_successful is not None:
        last_successful = last_successful.replace(
            microsecond=(last_successful.microsecond // 1000) * 1000
        )
    # Default the PK tiebreaker to 0 when no prior run exists (first full load)
    if last_successful_pk is None:
        last_successful_pk = 0

    print(f"\n=== Processing table: {table_name} ===")
    print(f"  last_ts : {last_successful}")
    print(f"  last_pk : {last_successful_pk}")

    # Step 2 — Read the source table; normalise its timestamp column to millisecond precision
    source_df = (
        spark.read.table(source_table)
        .withColumn(ts_col, F.date_trunc("millisecond", F.col(ts_col)).cast("timestamp"))
    )

    # Step 3 — Apply the incremental watermark filter
    # Full load  : no prior watermark → read every row from the source
    # Incremental: include rows where ts > last_ts,
    #              OR ts == last_ts AND pk > last_pk  (PK tiebreaker avoids re-loading boundary rows)
    if last_successful is None:
        rows_to_load = source_df                        # First run — no filter applied
    else:
        rows_to_load = source_df.filter(
            (F.col(ts_col) > F.lit(last_successful)) |
            ((F.col(ts_col) == F.lit(last_successful)) & (F.col(pk_col) > F.lit(last_successful_pk)))
        )

    # Step 4a — Attach Bronze audit columns to every qualifying row
    rows_to_load = (
        rows_to_load
        .withColumn("bronze_run_id",       F.lit(bronze_run_id))      # UUID of this pipeline run
        .withColumn("bronze_ingested_ts",  F.current_timestamp())     # Timestamp when the row was ingested
        .withColumn("bronze_source_table", F.lit(source_table))       # Source table name for lineage
    )

    rows_count = rows_to_load.count()               # Materialise to get the exact row count before writing
    print(f"  rows to load: {rows_count}")

    if rows_count == 0:
        print(f"  No new data for {table_name} — skipping.")  # Nothing new; move to the next table
        continue

    # Step 4b — Append qualifying rows to the Bronze Delta table
    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)

    # Step 5a — Compute the new watermark: maximum timestamp in this batch
    max_ts = rows_to_load.agg(F.max(ts_col).alias("max_ts")).collect()[0]["max_ts"]

    # Step 5b — Compute the PK tiebreaker: maximum PK among rows that share max_ts
    max_pk = (
        rows_to_load
        .filter(F.col(ts_col) == F.lit(max_ts))
        .agg(F.max(F.col(pk_col).cast("long")).alias("max_pk"))
        .collect()[0]["max_pk"]
    )

    # Step 5c — Persist the new watermark and run metadata to the ingestion control table
    upsert_bronze_control(table_name, ts_col, pk_col, max_ts, max_pk, rows_count, bronze_run_id)
    print(f"  Wrote {rows_count} rows → {target_table}")

In [0]:
# ── Post-load verification ───────────────────────────────────────────────────
# Print the total row count for each Bronze target table to confirm data was written
print("orders Bronze count :",   spark.sql("select count(*) from 00_bronze.orders_raw").collect()[0][0])
print("products Bronze count :", spark.sql("select count(*) from 00_bronze.products_raw").collect()[0][0])
print("payments Bronze count :", spark.sql("select count(*) from 00_bronze.payments_raw").collect()[0][0])

# Display the full ingestion control table sorted by table name.
# Review last_successful, last_successful_pk, and rows_written to confirm watermarks updated correctly.
display(spark.sql("select * from 00_bronze.ingestion_control").orderBy("table_name"))
